Forward propagation from scratch (NumPy)

Implement forward propagation manually to understand every operation

In [3]:
# ============================================================
# FORWARD PROPAGATION FROM SCRATCH
# No PyTorch/TensorFlow - pure NumPy to see the math!
# ============================================================


import numpy as np

# define the network

class NeuralNetwork:
  def __init__(self, layer_sizes):
    """
        layer_sizes: list of layer sizes [input, hidden1, hidden2, ..., output]
        Example: [4, 8, 6, 3] = 4 inputs, two hidden layers (8, 6 neurons), 3 outputs
    """
    self.layers = []

    # intialize weights and biases for each layer transition
    for i in range(len(layer_sizes) - 1):
      n_in = layer_sizes[i]
      n_out = layer_sizes[i+1]

      layer = {
          'W' : np.random.randn(n_out,n_in) * np.sqrt(2.0 / n_in),
          'b' : np.zeros((n_out,1))
      }

      self.layers.append(layer)
      print(f"Layer {i}: W shape  {layer['W'].shape}, b shape {layer['b'].shape}")

  def relu(self, z):
    return np.maximum(0,z)

  def softmax(self, z):
    """Numerically stable softmax for output layers"""
    # subtract max to prevent overflow (numerical stability trick)
    exp_z = np.exp(z - np.max(z, axis = 0))
    return exp_z / np.sum(exp_z, axis = 0)

  def forward(self, x, verbose = False):
    """Forward propagation through the network
    x: input vector (n_features, 1)
    Returns: output probabilities (n_classes , 1)"""

    activation = x

    if verbose:
      print(f"\n Input {activation.T}")

    for i, layer in enumerate(self.layers):
      # step 1 & 2: linear transformation z = W @ a + b
      z = layer['W'] @ activation + layer['b']

      # step 3: apply activation formula
      if i < len(self.layers) - 1:
        activation = self.relu(z) # relu for hidden layers
        if verbose:
          print(f"Layer {i + 1} (ReLU): {activation.T}")
      else:
        activation = self.softmax(z) # softmax for output layers
        if verbose:
          print(f"Output (Softmax): {activation.T}")
    return activation

net = NeuralNetwork([4,8,6,3])
x = np.array([[0.5], [0.8], [0.3], [0.1]])

output = net.forward(x, verbose = False)

print("\n" + "="*50)
print("FINAL PREDICTION")
print("="*50)
print(f"Class probabilities: {output.T}")
print(f"Sum of probabilities: {np.sum(output):.6f} (should be 1.0)")
print(f"Predicted class: {np.argmax(output)}")


# ============================================================
# COUNT PARAMETERS
# ============================================================
total_params = sum(
    layer['W'].size + layer['b'].size for layer in net.layers
)
print(f"\nTotal parameters: {total_params}")
print("Breakdown:")
for i, layer in enumerate(net.layers):
    w_params = layer['W'].size
    b_params = layer['b'].size
    print(f"  Layer {i}: {w_params} weights + {b_params} biases = {w_params + b_params}")




Layer 0: W shape  (8, 4), b shape (8, 1)
Layer 1: W shape  (6, 8), b shape (6, 1)
Layer 2: W shape  (3, 6), b shape (3, 1)

FINAL PREDICTION
Class probabilities: [[0.58089245 0.21092463 0.20818292]]
Sum of probabilities: 1.000000 (should be 1.0)
Predicted class: 0

Total parameters: 115
Breakdown:
  Layer 0: 32 weights + 8 biases = 40
  Layer 1: 48 weights + 6 biases = 54
  Layer 2: 18 weights + 3 biases = 21


Forward propagation in PyTorch (modern)

Use PyTorch's nn.Module for production-ready forward propagation




In [4]:
# ============================================================
# FORWARD PROPAGATION WITH PYTORCH
# Production-ready implementation with modern best practices
# ============================================================

import torch
import torch.nn as nn

# define the network

class SimpleNet(nn.Module):
  def __init__(self, input_dim, hidden_dims, output_dim):
    """
    input_dim - number of input features
    hidden_dims - number of hidden layers sizes [128,64]
    output_dim - number of output classes
    """
    super().__init__()

    # build layers dynamically
    layers = []
    prev_dim = input_dim

    for hidden_dim in hidden_dims:
      layers.append(nn.Linear(prev_dim, hidden_dim))
      layers.append(nn.ReLU())
      prev_dim = hidden_dim

    layers.append(nn.Linear(prev_dim, output_dim))
    # no softmax layer - pytorch's cross entropy loss
    # applies log_softmax internally for numerical stability

    self.network = nn.Sequential(*layers)

  def forward(self, x):
    """
    Forward pass through the network
    x - input tensor (batch_size, input_dim)
    returns logits (batch_size, output_dim)
    """
    return self.network(x)

# creat and test the model
# mnist style 784 inputs -> 128 -> 64 -> 10 digit classes

model = SimpleNet(input_dim=784, hidden_dims = [128,64], output_dim=10)
print("Model Architecture")
print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

batch_size = 4
x = torch.randn(batch_size, 784)

print(f"\nInput shape: {x.shape}")

# forward propagation
model.eval() # Disable dropout, batchnorm in eval mode

with torch.no_grad():  # no gradient tracking, faster + saves memory
  logits = model(x)
  probs = torch.softmax(logits, dim=1)
  predictions = torch.argmax(probs, dim=1)

print(f"Output logits shape: {logits.shape}")
print(f"\nProbabilities (first sample): \n{probs[0]}")
print(f"Sum: {probs[0].sum():.6f}")
print(f"Predictions: {predictions}")

# inspecting intermediate activations (debugging techniques)

print("\n" + "="*50)
print("INTERMEDIATE ACTIVATIONS")
print("="*50)

activations = {}

def get_activation(name):
  def hook(model, input, output):
    activations[name] = output.detach()
  return hook


# register hook on each relu layer (post activation, where no dead neuron matters)
for i, layer in enumerate(model.network):
  if isinstance(layer, nn.ReLU):
    layer.register_forward_hook(get_activation(f"relu_{i}"))

# run forward pass to capture activations
with torch.no_grad():
  _ = model(x)

# Print statistics - useful for debugging dead neurons or explosions
for name, act in activations.items():
    dead_pct = (act == 0).float().mean() * 100
    print(f"{name}: shape {act.shape}, "
          f"mean {act.mean():.4f}, std {act.std():.4f}, "
          f"dead neurons: {dead_pct:.1f}%")


Model Architecture
SimpleNet(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)
Total parameters: 109,386

Input shape: torch.Size([4, 784])
Output logits shape: torch.Size([4, 10])

Probabilities (first sample): 
tensor([0.1007, 0.1129, 0.0955, 0.0854, 0.1068, 0.0850, 0.0892, 0.0927, 0.1153,
        0.1166])
Sum: 1.000000
Predictions: tensor([9, 4, 8, 7])

INTERMEDIATE ACTIVATIONS
relu_1: shape torch.Size([4, 128]), mean 0.2421, std 0.3259, dead neurons: 47.1%
relu_3: shape torch.Size([4, 64]), mean 0.1019, std 0.1480, dead neurons: 46.9%


Batch processing and GPU acceleration

See the performance impact of batching and GPU computation




In [5]:
# ============================================================
# BATCH PROCESSING & GPU ACCELERATION
# See the performance difference between batch sizes
# ============================================================

import torch
import torch.nn as nn
import time

# ============================================================
# CREATE A LARGER MODEL
# ============================================================
class LargeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(784, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.network(x)

model = LargeNet()
print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters")

# ============================================================
# BENCHMARK: BATCH SIZE IMPACT ON THROUGHPUT
# ============================================================

def benchmark_forward(model, batch_size, device='cpu', num_iterations=100):
    """Time forward propagation for different batch sizes"""
    model = model.to(device)
    x = torch.randn(batch_size, 784, device=device)

    # Warmup (important for accurate timing)
    for _ in range(10):
        with torch.no_grad():
            _ = model(x)
    if device != 'cpu':
        torch.cuda.synchronize()

    # Benchmark
    start = time.time()
    for _ in range(num_iterations):
        with torch.no_grad():
            _ = model(x)
    if device != 'cpu':
        torch.cuda.synchronize()
    end = time.time()

    return (end - start) / num_iterations

print("\n" + "="*50)
print("CPU PERFORMANCE (batch size impact)")
print("="*50)

batch_sizes = [1, 8, 32, 128]
for bs in batch_sizes:
    time_per_batch = benchmark_forward(model, bs, 'cpu', num_iterations=50)
    time_per_sample = time_per_batch / bs
    samples_per_sec = 1.0 / time_per_sample
    print(f"Batch {bs:3d}: {time_per_batch*1000:.2f}ms/batch, "
          f"{time_per_sample*1000:.3f}ms/sample, "
          f"{samples_per_sec:.0f} samples/sec")

# ============================================================
# GPU ACCELERATION (if available)
# ============================================================

if torch.cuda.is_available():
    print("\n" + "="*50)
    print(f"GPU PERFORMANCE ({torch.cuda.get_device_name(0)})")
    print("="*50)

    for bs in [1, 32, 128, 512]:
        time_per_batch = benchmark_forward(
            LargeNet(), bs, 'cuda', num_iterations=100
        )
        time_per_sample = time_per_batch / bs
        samples_per_sec = 1.0 / time_per_sample
        print(f"Batch {bs:3d}: {time_per_batch*1000:.2f}ms/batch, "
              f"{time_per_sample*1000:.3f}ms/sample, "
              f"{samples_per_sec:.0f} samples/sec")
else:
    print("\nNo GPU available. Install CUDA for GPU acceleration.")

# ============================================================
# KEY TAKEAWAYS
# ============================================================
print("\n" + "="*50)
print("KEY TAKEAWAYS")
print("="*50)
print("1. Larger batches = better GPU utilization (up to memory limit)")
print("2. GPU speedup is 10-100x for large models and batches")
print("3. Time per sample DECREASES as batch size increases")
print("4. Always use model.eval() + torch.no_grad() for inference")
print("5. Batch size 1 wastes GPU - use 32-512 for training")

Model has 1,462,538 parameters

CPU PERFORMANCE (batch size impact)
Batch   1: 0.50ms/batch, 0.501ms/sample, 1998 samples/sec
Batch   8: 1.12ms/batch, 0.140ms/sample, 7143 samples/sec
Batch  32: 2.46ms/batch, 0.077ms/sample, 12986 samples/sec
Batch 128: 7.77ms/batch, 0.061ms/sample, 16469 samples/sec

No GPU available. Install CUDA for GPU acceleration.

KEY TAKEAWAYS
1. Larger batches = better GPU utilization (up to memory limit)
2. GPU speedup is 10-100x for large models and batches
3. Time per sample DECREASES as batch size increases
4. Always use model.eval() + torch.no_grad() for inference
5. Batch size 1 wastes GPU - use 32-512 for training
